# 05 — The artificial experiments of section 6.1


In [ ]:
# --- make src/ importable: src/common + the experiment folder --------------
import os, sys
from pathlib import Path
_ROOT = Path.cwd()
while not (_ROOT / "src").is_dir() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
sys.path[:0] = [str(_ROOT / "src" / "common"), str(_ROOT / "src" / "artificial_graphs")]

import time, itertools
import numpy as np
import pandas as pd

import graph_learning as gl
import smooth_signals as ss
import metrics as mt
import paths
import run_experiments_artificial_graphs as rea   # the driver, src/artificial_graphs/

M        = rea.M        # 100 nodes
N_SIG    = 1000         # signals (Table 2);  Table 4 of the appendix uses 100
NOISE    = rea.NOISE    # 10% in the l-2 sense
N_TRIALS = rea.N_TRIALS # the paper uses 20
N_GRID   = rea.N_GRID   # grid points per model
EPS      = rea.EPS      # solver tolerance; see metrics.EDGE_TOL for why
MAXIT    = rea.MAX_ITER

BETAS, ALPHAS, SIGMAS = rea._grids(N_GRID)   # (12): sweep beta / (14): sweep alpha
                                             # / baseline eq. (11): sweep sigma
GRAPH_NAMES  = list(ss.GRAPHS)
FILTER_NAMES = list(ss.FILTERS)
METRICS      = list(mt.ARTIFICIAL_METRICS)

print(f"{len(GRAPH_NAMES)} graphs x {len(FILTER_NAMES)} filters x {N_TRIALS} trials "
      f"x {N_GRID} grid points x 2 models = "
      f"{len(GRAPH_NAMES)*len(FILTER_NAMES)*N_TRIALS*N_GRID*2} optimizations")
print("cores:", os.cpu_count(), "| repo root:", _ROOT)


## One trial


In [ ]:
def one_trial(graph_name, filter_name, seed, n_sig=N_SIG):
    return rea.one_trial(graph_name, filter_name, seed, n_sig, n_grid=N_GRID)


_probe = one_trial("Rand. Geometric", "Tikhonov", 0, n_sig=200)
print(f"probe: {len(_probe)} records")
pd.DataFrame(_probe).query("model == 'Ours'")[["param", "density", "n_components"] + METRICS].round(3)


## Run everything


In [ ]:
def run_all(n_sig, n_trials=N_TRIALS, tag=""):
    records, t0 = [], time.perf_counter()
    todo = list(itertools.product(GRAPH_NAMES, FILTER_NAMES))
    for k, (g, f) in enumerate(todo, 1):
        for t in range(n_trials):
            records += one_trial(g, f, seed=1000*k + t, n_sig=n_sig)
        el = time.perf_counter() - t0
        print(f"[{tag}] {k:2d}/{len(todo)}  {g:17s} {f:17s} "
              f"{el:6.1f}s elapsed, ~{el/k*(len(todo)-k):5.1f}s left")
    return pd.DataFrame(records)


DF1000 = run_all(N_SIG, tag="n=1000")
print(f"\n{len(DF1000)} records")
print("non-converged runs:",
      (~DF1000.converged).sum(), "/", len(DF1000))


## Table 2


In [ ]:
def summarize(df):
    vals, pars = {}, {}
    for (g, f, mo), sub in df.groupby(["graph", "filter", "model"], sort=False):
        best = mt.best_parameter_per_metric(sub.to_dict("records"),
                                            param_keys=("param",))
        vals[(g, f, mo)] = {k: best[k]["value"] for k in METRICS}
        pars[(g, f, mo)] = {k: best[k]["param"] for k in METRICS}
    idx = pd.MultiIndex.from_tuples(vals, names=["graph", "filter", "model"])
    return (pd.DataFrame(list(vals.values()), index=idx),
            pd.DataFrame(list(pars.values()), index=idx))


VALS1000, PARS1000 = summarize(DF1000)

def as_table2(vals):
    out = (vals.stack().rename("v").reset_index()
              .rename(columns={"level_3": "metric"})
              .pivot(index=["graph", "metric"], columns=["filter", "model"],
                     values="v")
              .reindex(columns=pd.MultiIndex.from_product(
                  [FILTER_NAMES, ["base", "Dong etal", "Ours"]],
                  names=["filter", "model"]))
              .reindex(index=pd.MultiIndex.from_product([GRAPH_NAMES, METRICS],
                                                        names=["graph", "metric"])))
    return out.round(3)

TABLE2 = as_table2(VALS1000)
TABLE2


### Against the published Table 2


In [ ]:
# Table 2 of the paper, transcribed.  Order: F, edge l-1, edge l-2, deg l-1, deg l-2
PAPER_T2 = {
 ("Rand. Geometric", "Tikhonov"):        {"base":[.685,.866,.676,.142,.708], "Dong etal":[.885,.357,.376,.146,.172], "Ours":[.913,.298,.336,.065,.079]},
 ("Rand. Geometric", "Generative Model"):{"base":[.686,.798,.658,.261,.689], "Dong etal":[.877,.371,.397,.147,.174], "Ours":[.909,.348,.390,.112,.128]},
 ("Rand. Geometric", "Heat Diffusion"):  {"base":[.758,.609,.576,.209,.474], "Dong etal":[.837,.524,.531,.227,.264], "Ours":[.849,.447,.468,.142,.176]},
 ("Non Uniform", "Tikhonov"):            {"base":[.686,.821,.706,.160,.612], "Dong etal":[.863,.423,.434,.184,.209], "Ours":[.858,.349,.344,.055,.073]},
 ("Non Uniform", "Generative Model"):    {"base":[.633,.864,.735,.235,.632], "Dong etal":[.840,.487,.480,.185,.215], "Ours":[.832,.472,.474,.100,.161]},
 ("Non Uniform", "Heat Diffusion"):      {"base":[.766,.594,.550,.233,.427], "Dong etal":[.839,.565,.587,.255,.324], "Ours":[.830,.473,.451,.128,.157]},
 ("Erdos Renyi", "Tikhonov"):            {"base":[.288,1.465,1.060,.094,.986],"Dong etal":[.766,.448,.442,.107,.161], "Ours":[.893,.391,.402,.046,.066]},
 ("Erdos Renyi", "Generative Model"):    {"base":[.199,1.566,1.105,.099,1.312],"Dong etal":[.755,.478,.457,.105,.181],"Ours":[.896,.427,.440,.066,.151]},
 ("Erdos Renyi", "Heat Diffusion"):      {"base":[.377,1.379,1.033,.182,.892],"Dong etal":[.629,.832,.735,.179,.236], "Ours":[.655,.841,.726,.183,.273]},
 ("Barabasi-Albert", "Tikhonov"):        {"base":[.345,1.531,1.061,.175,.554],"Dong etal":[.710,.614,.568,.264,.340], "Ours":[.868,.533,.506,.111,.201]},
 ("Barabasi-Albert", "Generative Model"):{"base":[.382,1.496,1.036,.199,.556],"Dong etal":[.739,.652,.611,.264,.333], "Ours":[.838,.624,.571,.207,.287]},
 ("Barabasi-Albert", "Heat Diffusion"):  {"base":[.352,1.468,1.041,.254,.568],"Dong etal":[.690,.740,.662,.317,.414], "Ours":[.765,.675,.590,.148,.283]},
}
PAPER = pd.DataFrame(
    {(g, f, mo): dict(zip(METRICS, v)) for (g, f), d in PAPER_T2.items()
     for mo, v in d.items()}).T
PAPER.index.names = ["graph", "filter", "model"]

cmp_rows = []
for (g, f), d in PAPER_T2.items():
    for metric in METRICS:
        bigger = mt.ARTIFICIAL_METRICS[metric]
        ours_v = {mo: VALS1000.loc[(g, f, mo), metric] for mo in ("base", "Dong etal", "Ours")}
        pap_v  = {mo: PAPER.loc[(g, f, mo), metric]    for mo in ("base", "Dong etal", "Ours")}
        pick = (max if bigger else min)
        cmp_rows.append({
            "graph": g, "filter": f, "metric": metric,
            "paper winner": pick(pap_v, key=pap_v.get),
            "our winner":   pick(ours_v, key=ours_v.get),
            "paper Ours": pap_v["Ours"], "our Ours": round(ours_v["Ours"], 3),
            "paper Dong": pap_v["Dong etal"], "our Dong": round(ours_v["Dong etal"], 3),
        })
CMP = pd.DataFrame(cmp_rows)
CMP["same winner"] = CMP["paper winner"] == CMP["our winner"]
print(f"same winning model as the paper in "
      f"{CMP['same winner'].sum()} / {len(CMP)} (graph, filter, metric) cells")
print(f"'Ours' wins in {(CMP['our winner']=='Ours').sum()}/{len(CMP)} of our cells, "
      f"{(CMP['paper winner']=='Ours').sum()}/{len(CMP)} of the paper's")
CMP


In [ ]:
# how far are the absolute numbers?
delta = (VALS1000 - PAPER).dropna()
print("ours - paper, averaged over the 12 settings:")
print(delta.groupby("model").mean().round(3))
print("\nmean |ours - paper| per metric:")
print(delta.abs().groupby("model").mean().round(3))


### Picture


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(METRICS), 1, figsize=(15, 3.1*len(METRICS)), sharex=True)
labels = [f"{g.split()[0][:9]}\n{f.split()[0][:6]}" for g in GRAPH_NAMES
          for f in FILTER_NAMES]
xpos = np.arange(len(labels))
colors = {"base": "0.72", "Dong etal": "tab:orange", "Ours": "tab:blue"}

for ax, metric in zip(axes, METRICS):
    for k, mo in enumerate(["base", "Dong etal", "Ours"]):
        ours = [VALS1000.loc[(g, f, mo), metric] for g in GRAPH_NAMES for f in FILTER_NAMES]
        pap  = [PAPER.loc[(g, f, mo), metric]    for g in GRAPH_NAMES for f in FILTER_NAMES]
        ax.bar(xpos + (k-1)*0.26, ours, width=0.24, color=colors[mo],
               label=mo if metric == METRICS[0] else None)
        ax.plot(xpos + (k-1)*0.26, pap, "k_", ms=11, mew=1.8,
                label="paper" if (metric == METRICS[0] and k == 2) else None)
    ax.set_ylabel(metric)
    ax.grid(axis="y", alpha=0.3)
    for b in range(1, 4):
        ax.axvline(3*b - 0.5, color="k", lw=0.6, alpha=0.4)
# headroom so the legend sits above the bars instead of on top of them
axes[0].set_ylim(0, 1.30)
axes[0].legend(ncol=4, fontsize=9, loc="upper left", framealpha=0.95)
axes[0].set_title(f"section 6.1 reproduced  (m = {M}, n = {N_SIG}, "
                  f"{N_TRIALS} trials)   bars = this notebook, "
                  f"black dashes = Table 2 of the paper", fontsize=12)
axes[-1].set_xticks(xpos, labels, fontsize=8)
plt.tight_layout(); plt.show()


### The parameters that were selected


In [ ]:
GRIDS = {"Ours": BETAS, "Dong etal": ALPHAS, "base": SIGMAS}
edge_hits = []
for (g, f, mo), row in PARS1000.iterrows():
    grid = GRIDS[mo]
    for metric in METRICS:
        if np.isclose(row[metric], grid[0]) or np.isclose(row[metric], grid[-1]):
            edge_hits.append({"graph": g, "filter": f, "model": mo,
                              "metric": metric, "param": row[metric],
                              "end": "low" if np.isclose(row[metric], grid[0])
                                     else "high"})
HITS = pd.DataFrame(edge_hits)
print(f"grid-endpoint selections: {len(HITS)} / {PARS1000.size}")
if len(HITS):
    print(HITS.groupby(["model", "metric"]).size().rename("hits").to_string())
print("""
Reading this: almost all hits are the baseline's F-measure. That one is expected
and harmless -- the baseline F-measure is taken at the BEST THRESHOLD of
exp(-z/2sigma^2), and thresholding a monotone function of z gives the same binary
pattern whatever sigma is, so the F-measure is constant along the sigma grid and
`max` simply returns the first point. A hit for "Ours" or "Dong etal", on the
other hand, means the grid should be widened on that side.""")
print(HITS[HITS.model != "base"].to_string(index=False) if len(HITS) else "")
PARS1000.round(4)


## Table 4 — the same with $n=100$ signals


In [ ]:
RUN_TABLE4 = True

if RUN_TABLE4:
    DF100 = run_all(100, tag="n=100")
    VALS100, PARS100 = summarize(DF100)
    TABLE4 = as_table2(VALS100)
else:
    TABLE4 = None
TABLE4


In [ ]:
if RUN_TABLE4:
    PAPER_T4 = {
     ("Rand. Geometric","Tikhonov"):        {"base":[.667,.896,.700,.158,.707],"Dong etal":[.860,.414,.430,.151,.179],"Ours":[.886,.364,.390,.080,.095]},
     ("Rand. Geometric","Generative Model"):{"base":[.671,.851,.692,.268,.679],"Dong etal":[.836,.487,.494,.159,.193],"Ours":[.858,.468,.477,.128,.145]},
     ("Rand. Geometric","Heat Diffusion"):  {"base":[.752,.620,.582,.216,.479],"Dong etal":[.837,.526,.535,.225,.264],"Ours":[.848,.451,.471,.143,.177]},
     ("Non Uniform","Tikhonov"):            {"base":[.674,.847,.724,.167,.605],"Dong etal":[.821,.547,.545,.190,.228],"Ours":[.817,.480,.462,.075,.099]},
     ("Non Uniform","Generative Model"):    {"base":[.650,.931,.784,.241,.614],"Dong etal":[.779,.711,.673,.204,.261],"Ours":[.774,.673,.624,.139,.187]},
     ("Non Uniform","Heat Diffusion"):      {"base":[.763,.612,.565,.235,.433],"Dong etal":[.835,.583,.598,.257,.325],"Ours":[.827,.491,.464,.132,.164]},
     ("Erdos Renyi","Tikhonov"):            {"base":[.293,1.513,1.086,.114,.932],"Dong etal":[.595,.837,.712,.129,.202],"Ours":[.676,.798,.697,.084,.116]},
     ("Erdos Renyi","Generative Model"):    {"base":[.207,1.623,1.129,.135,1.053],"Dong etal":[.473,1.113,.896,.146,.227],"Ours":[.512,1.090,.888,.114,.185]},
     ("Erdos Renyi","Heat Diffusion"):      {"base":[.358,1.401,1.045,.185,.875],"Dong etal":[.595,.896,.767,.182,.241],"Ours":[.619,.899,.759,.184,.276]},
     ("Barabasi-Albert","Tikhonov"):        {"base":[.325,1.541,1.073,.225,.560],"Dong etal":[.564,.939,.802,.309,.378],"Ours":[.636,.885,.761,.145,.281]},
     ("Barabasi-Albert","Generative Model"):{"base":[.357,1.513,1.052,.243,.563],"Dong etal":[.588,.940,.808,.311,.386],"Ours":[.632,.914,.773,.229,.350]},
     ("Barabasi-Albert","Heat Diffusion"):  {"base":[.349,1.473,1.049,.281,.570],"Dong etal":[.631,.843,.732,.336,.429],"Ours":[.711,.774,.672,.181,.319]},
    }
    PAPER4 = pd.DataFrame(
        {(g, f, mo): dict(zip(METRICS, v)) for (g, f), d in PAPER_T4.items()
         for mo, v in d.items()}).T
    PAPER4.index.names = ["graph", "filter", "model"]
    wins = sum(
        (max if mt.ARTIFICIAL_METRICS[me] else min)(
            {mo: VALS100.loc[(g, f, mo), me] for mo in ("base","Dong etal","Ours")},
            key=lambda mo: VALS100.loc[(g, f, mo), me]) ==
        (max if mt.ARTIFICIAL_METRICS[me] else min)(
            {mo: PAPER4.loc[(g, f, mo), me] for mo in ("base","Dong etal","Ours")},
            key=lambda mo: PAPER4.loc[(g, f, mo), me])
        for (g, f) in PAPER_T4 for me in METRICS)
    print(f"n = 100: same winner as the paper in {wins} / {12*len(METRICS)} cells")
    print("\nours - paper, averaged over the 12 settings:")
    print((VALS100 - PAPER4).dropna().groupby("model").mean().round(3))


In [ ]:
# n = 1000 vs n = 100, our own numbers: does the gap really widen?
if RUN_TABLE4:
    manifold = ["Rand. Geometric", "Non Uniform"]
    rand     = ["Erdos Renyi", "Barabasi-Albert"]
    rows = []
    for lab, vals in [("n = 1000", VALS1000), ("n = 100", VALS100)]:
        for gl_lab, gs in [("manifold", manifold), ("non-manifold", rand)]:
            sub = vals.loc[[(g, f, "Ours") for g in gs for f in FILTER_NAMES]]
            rows.append({"n": lab, "graphs": gl_lab,
                         **sub.mean().round(3).to_dict()})
    print("model (12) averaged over the 3 filters:")
    display(pd.DataFrame(rows).set_index(["n", "graphs"]))


## Reading the results
